In [ ]:
import pymilvus
print(pymilvus.__version__)  # 应该输出 2.5.18


In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv


load_dotenv()

llm=ChatOpenAI(model="deepseek-chat",
               base_url=os.getenv("DEEPSEEK_BASE_URL"),
               api_key=os.getenv("DEEPSEEK_API_KEY"),
               temperature=0)

In [ ]:
# 打开文件，并赋予读取模式
with open("./company.txt",'r',encoding='utf-8') as file:
    content=file.read()
    print(content)

In [ ]:
from langchain_core.documents import Document

documents=[Document(page_content=content)]

In [ ]:
documents

In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

print(os.getenv("NEO4J_URI"))
print(os.getenv("NEO4J_USERNAME"))
print(os.getenv("NEO4J_PASSWORD"))
print(os.getenv("NEO4J_DATABASE"))





In [ ]:
graph=Neo4jGraph(url=os.getenv("NEO4J_URI"),
                 username=os.getenv("NEO4J_USERNAME"),
                 password=os.getenv("NEO4J_PASSWORD"),
                 database=os.getenv("NEO4J_DATABASE"))

In [ ]:
# 图转换器配置
graph_transformer=LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["公司","产品","技术","市场","活动","合作伙伴"],
    allowed_relationships=["推出","参与","合作","位于","开发"]
)

graph_transformer=LLMGraphTransformer(llm=llm,ignore_tool_usage=True)

graph_documents=graph_transformer.convert_to_graph_documents(documents)

graph.add_graph_documents(graph_documents)


print(f"Graph documents: {len(graph_documents)}")
print(f"Nodes from 1st graph doc: {graph_documents[0].nodes}")
print(f"Relationships from 1st graph doc: {graph_documents[0].relationships}")

In [ ]:
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

# 自定义 Cypher 生成 prompt - 加强约束
# --- 自定义 prompt：强调用精确 ID + CONTAINS 兜底 ---
CUSTOM_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""Task: Generate ONE Cypher query to answer the question about a Chinese company.

CRITICAL RULES:
1. Identify the entity keyword from the question (e.g. "小米", "苹果", "华为")
2. ALWAYS use CONTAINS for text matching — NEVER use = for Chinese company names
   Example: WHERE n.id CONTAINS '小米'  (NOT: WHERE n.id = '小米公司')
3. Match ONLY the keyword part (before 公司/科技/技术), because the full name in the database may differ
4. Return ONLY the Cypher statement. No markdown, no explanation, no ```cypher```.

Schema:
{schema}

Question: {question}

Cypher:"""
)

cypher_chain=GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=llm,
    cypher_prompt=CUSTOM_PROMPT,
    qa_llm=llm,
    validate_cypher=True,
    verbose=True,
    allow_dangerous_requests=True
)

cypher_chain.invoke("苹果公司开发了什么")



In [ ]:
cypher_chain.invoke("都有哪些公司在数据库中")

In [ ]:
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os; load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat",
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0
)

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE")
)

# --- 构建增强版 Schema：包含实际节点 ID ---
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)
with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
    result = session.run(
        "MATCH (n) RETURN DISTINCT labels(n)[0] as label, "
        "collect(DISTINCT n.id)[..15] as sample_ids"
    )
    enriched = graph.schema + "\n\n=== 实际节点ID（查询时必须用这些精确值）===\n"
    for r in result:
        enriched += f"{r['label']}: {r['sample_ids']}\n"
driver.close()

graph.schema = enriched  # 替换默认 schema

# --- 自定义 prompt：强调用精确 ID + CONTAINS 兜底 ---
CUSTOM_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""Task: Generate ONE Cypher query to answer the question about a Chinese company.

CRITICAL RULES:
1. Identify the entity keyword from the question (e.g. "小米", "苹果", "华为")
2. ALWAYS use CONTAINS for text matching — NEVER use = for Chinese company names
   Example: WHERE n.id CONTAINS '小米'  (NOT: WHERE n.id = '小米公司')
3. Match ONLY the keyword part (before 公司/科技/技术), because the full name in the database may differ
4. Return ONLY the Cypher statement. No markdown, no explanation, no ```cypher```.

Schema:
{schema}

Question: {question}

Cypher:"""
)


chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=CUSTOM_PROMPT,
    verbose=True,
    allow_dangerous_requests=True,
)

chain.invoke("介绍一下小米公司")


In [ ]:
chain.invoke("介绍一下苹果公司")

In [ ]:
# chain.invoke("苹果公司开发了什么")

In [ ]:
from langgraph.graph import StateGraph,MessagesState ,START,END

class AgentState(MessagesState):
    next:str

In [ ]:
from langchain.messages import HumanMessage

def graph_kg(state:AgentState):
    messages=state["messages"][-1]
    
    # 自定义 Cypher 生成 prompt - 加强约束
    # --- 自定义 prompt：强调用精确 ID + CONTAINS 兜底 ---
    CUSTOM_PROMPT = PromptTemplate(
        input_variables=["schema", "question"],
        template="""Task: Generate ONE Cypher query to answer the question about a Chinese company.

    CRITICAL RULES:
    1. Identify the entity keyword from the question (e.g. "小米", "苹果", "华为")
    2. ALWAYS use CONTAINS for text matching — NEVER use = for Chinese company names
    Example: WHERE n.id CONTAINS '小米'  (NOT: WHERE n.id = '小米公司')
    3. Match ONLY the keyword part (before 公司/科技/技术), because the full name in the database may differ
    4. Return ONLY the Cypher statement. No markdown, no explanation, no ```cypher```.

    Schema:
    {schema}

    Question: {question}

    Cypher:"""
    )

    cypher_chain=GraphCypherQAChain.from_llm(
        graph=graph,
        cypher_llm=llm,
        cypher_prompt=CUSTOM_PROMPT,
        qa_llm=llm,
        validate_cypher=True,
        verbose=True,
        allow_dangerous_requests=True
    )
    response=cypher_chain.invoke(messages.content)
    final_response=[HumanMessage(content=response['result'],name="graph_kg")]
    return {"messages":final_response}

In [ ]:
# 使用向量数据库
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size=250
chunk_overlap=30
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,chunk_overlap=chunk_overlap
)

splits=text_splitter.split_documents(documents)
splits

In [ ]:
from langchain_ollama import OllamaEmbeddings

# DeepSeek 没有 Embedding API（/v1/embeddings 不存在，404）
# 改用本地 Ollama 的 bge-m3 模型 —— 支持中英双语，1024维向量
embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

pip install langchain_milvus

In [ ]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_milvus import Milvus

# vectorstore=Milvus.from_documents(
#     documents=splits,
#     collection_name="compnay_rag_milvus",
#     embedding=embeddings,
#     connection_args={
#         "uri":os.getenv("zilliz_url"),
#         "user":os.getenv("zilliz_User"),
#         "password":os.getenv("zilliz_Password")
#     }
# )

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_milvus import Milvus
# 将切割好的数据，插入到数据库中
vectorstore = Milvus.from_documents(
    documents=splits,
    collection_name="company_rag_milvus",
    embedding=embeddings,
    drop_old=True,   # ← 关键！删除之前错误 schema 的旧 collection
    connection_args={
        "uri": os.getenv("zilliz_url"),
        "token": f"{os.getenv('zilliz_User')}:{os.getenv('zilliz_Password')}",
    }
)


In [ ]:

from langchain_core.output_parsers import StrOutputParser

prompt=PromptTemplate(
    template="""
        you are an assistant for question-answering tasks.
        use the following pieces of retrieved context to answer the question. If you don't know the answer, just 
        use three sentences maximum and keep the answer concise:
        Question: {question}
        Context: {context}
        Answer:
    """,
    input_variables={"question",'context'}
)

rag_chain=prompt | llm | StrOutputParser()

question="我的知识库中都有哪些公司信息"

retriever=vectorstore.as_retriever(search_kwargs={"k":1})

docs=retriever.invoke("question")

docs

In [ ]:
generation=rag_chain.invoke({"context":docs,"question":question})
print(generation)

In [ ]:
def vec_kg(state:AgentState):
    
    messages=state["messages"][-1]
    question=messages.content
    
    prompt=PromptTemplate(
        template="""
            you are an assistant for question-answering tasks.
            use the following pieces of retrieved context to answer the question. If you don't know the answer, just 
            use three sentences maximum and keep the answer concise:
            Question: {question}
            Context: {context}
            Answer:
        """,
        input_variables={"question",'context'}
    )
    
    rag_chain=prompt | llm | StrOutputParser()
    retriever=vectorstore.as_retriever(search_kwargs={"k":1})
    docs=retriever.invoke("question")
    generation=rag_chain.invoke({"context":docs,"question":question})
    final_response=[HumanMessage(content=generation,name="vec_kg")]
    return {"messages":final_response}
    